# MODFLOW layer-1 heads vs D-Flow FM discretization

The D-Flow FM grid sets the water levels handed to the MODFLOW coastal boundary,
so refining it changes simulated groundwater levels even though the MODFLOW grid
itself never changes. That difference is what drives the sewer exchange apart
between grids: at 8-hour coupling the net infiltration falls from 187 to 83
gpd/in-diameter/mile going from coarse to midres, because fewer junctions reach
their pipe inverts and the driving head at those that do is smaller.

This notebook quantifies the head differences directly.

**All scenarios share the same MODFLOW grid** (`gp_chd`, 16 layers, 48 x 58), so
heads are compared cell by cell with no regridding &mdash; only the D-Flow grid
feeding the boundary differs.

Absolute head maps look nearly identical across grids, because they all span the
same few feet in a similar pattern, so the 9-panel figure shows the **reference
grid's heads on the top row and DIFFERENCES on the lower two**, which is where
the spatial structure of the disagreement actually shows up.

In [ ]:
%matplotlib inline
import sys
import pathlib as pl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import flopy
import flopy.plot.styles as styles

In [ ]:
sys.path.append("../common")
from liss_settings import (
    get_scenario_name,
    get_modflow_run_path,
    fig_ext,
    transparent,
)
from swmm_mf_connect import intersect_points_grid

#### Configuration

In [ ]:
# ---- held fixed; only the D-Flow FM grid varies ----------------------------
domain = "gp"
boundary_condition = "chd"
mf_couple_freq_hours = 8.0        # compare grids at MATCHED coupling
n_connections = 244
# ----------------------------------------------------------------------------

# coarsest first; the LAST one present is used as the reference
resolutions = ["coarse", "medium", "high"]
res_label = {"coarse": "coarse", "medium": "midres", "high": "highres"}

LAYER = 0                          # layer 1 (0-based)
target_months = [1, 2, 3]          # snapshot times, in months of simulated time
HDRY = 1e30                        # MODFLOW dry/inactive marker

fig_ws = pl.Path("figures")
fig_ws.mkdir(exist_ok=True, parents=True)

#### Load layer-1 heads

Heads live in the scenario MODFLOW run directory rather than the results
directory &mdash; step2 leaves them in place because `gwf.cbc` alone is ~184 MB.
Grids that have not been run are reported and skipped.

In [ ]:
heads = {}
for res in resolutions:
    run = get_modflow_run_path(domain, boundary_condition, res,
                               mf_couple_freq_hours, n_connections)
    hds = run / "gwf.hds"
    if not hds.is_file():
        print(f"  skipping {res_label[res]} - not run yet ({run})")
        continue
    h = flopy.utils.HeadFile(hds)
    times = np.array(h.get_times())
    arr = np.stack([h.get_data(totim=t)[LAYER] for t in times])   # (ntime, nrow, ncol)
    arr = np.where(np.abs(arr) >= 0.5 * HDRY, np.nan, arr)        # mask dry/inactive
    heads[res] = {"t": times, "h": arr, "run": run}
    print(f"  {res_label[res]:>8s}: {arr.shape[0]:4d} times, grid {arr.shape[1]}x{arr.shape[2]}, "
          f"head {np.nanmin(arr):6.2f}..{np.nanmax(arr):5.2f} ft, "
          f"{int(np.isnan(arr[0]).sum()):,} inactive cells")

assert heads, "no scenarios have been run at this coupling frequency"
have = [r for r in resolutions if r in heads]
ref = have[-1]                      # finest available grid is the reference
print(f"\n  reference grid: {res_label[ref]}")

# every scenario must share the MODFLOW grid, or a cell-by-cell diff is meaningless
shapes = {r: heads[r]["h"].shape[1:] for r in have}
assert len(set(shapes.values())) == 1, f"MODFLOW grids differ between scenarios: {shapes}"
ntime = min(heads[r]["h"].shape[0] for r in have)
print(f"  common time steps: {ntime}")

#### Snapshot indices at 1, 2 and 3 months

In [ ]:
t_ref = heads[ref]["t"][:ntime]
span = t_ref[-1]

# A run still in progress has fewer saved times than the others, which would
# silently collapse every snapshot onto its last step and compare only the part
# it has reached. Say so rather than quietly reporting a 12-day comparison as
# though it covered three months.
lens = {res_label[r]: heads[r]["h"].shape[0] for r in have}
if len(set(lens.values())) > 1:
    print(f"  WARNING: scenarios have different numbers of saved times: {lens}")
    print(f"           comparing only the first {ntime} steps "
          f"(through day {span:.1f}) that all of them share.")
    if span < max(target_months) * 30.0:
        print(f"           the requested {max(target_months)}-month snapshot is NOT "
              f"covered - a run is probably still in progress.")

snap_idx, snap_lbl = [], []
for m in target_months:
    want = m * 30.0
    if want > span:                       # not reached yet: fall back to the last step
        i = ntime - 1
    else:
        i = int(np.argmin(np.abs(t_ref - want)))
    if i in snap_idx:                     # never plot the same step twice
        continue
    snap_idx.append(i)
    lbl = f"day {t_ref[i]:.0f}"
    if abs(t_ref[i] - want) < 3.0:
        lbl = f"{m} month{'s' if m > 1 else ''}  ({lbl})"
    snap_lbl.append(lbl)

# if snapshots collapsed, spread them evenly over whatever record exists
if len(snap_idx) < len(target_months):
    snap_idx = sorted({int(round(f * (ntime - 1)))
                       for f in np.linspace(1.0 / len(target_months), 1.0, len(target_months))})
    snap_lbl = [f"day {t_ref[i]:.0f}" for i in snap_idx]
    print("  snapshots collapsed onto one step; spread evenly over the common record instead")

for i, l in zip(snap_idx, snap_lbl):
    print(f"  index {i:4d} -> {l}  (t = {t_ref[i]:.2f} d)")

#### Sewer junctions and the zoom window

The 244 connections sit in the Greenport village area on the North Fork.
Because the peninsula is narrow they are essentially all coastal-adjacent: the
median distance to the nearest CHD or GHB cell is 2 cells (~1,000 ft), 55% are
within 2 cells and 97% within 5. They sit right in the zone where the D-Flow
boundary forcing has most influence, which is why the grid choice moves the
exchange so strongly.

They occupy only 82 of 1,454 active layer-1 cells, so a full-domain map spends
~95% of its area on cells the sewer never touches. The zoomed figure pads the
junction bounding box slightly so surrounding boundary cells stay visible.

In [ ]:
n_conn, junctions, mf6_cells, swmm_inverts, _ = intersect_points_grid(
    domain=domain, boundary_condition=boundary_condition, n_junctions=500)
j_row = np.array([mf6_cells[j][1] for j in junctions])
j_col = np.array([mf6_cells[j][2] for j in junctions])
j_inv = np.array([swmm_inverts[j] for j in junctions])       # already in FEET

PAD = 2
nr, nc = heads[ref]["h"].shape[1:]
r0, r1 = max(j_row.min() - PAD, 0), min(j_row.max() + PAD, nr - 1)
c0, c1 = max(j_col.min() - PAD, 0), min(j_col.max() + PAD, nc - 1)

j_mask = np.zeros((nr, nc), dtype=bool)
j_mask[j_row, j_col] = True

print(f"  {n_conn} junctions in {int(j_mask.sum())} unique cells")
print(f"  junction extent : rows {j_row.min()}-{j_row.max()}, cols {j_col.min()}-{j_col.max()}")
print(f"  zoom window     : rows {r0}-{r1}, cols {c0}-{c1} (pad {PAD})")

#### Nine-panel figure

Top row: the reference grid's layer-1 heads. Lower rows: each coarser grid MINUS
the reference, on a symmetric diverging scale, so blue means that grid simulates
lower heads than the reference and red means higher.

In [ ]:
others = [r for r in have if r != ref]
nrow = 1 + len(others)

# symmetric color limits from the actual differences, so zero is white
dmax = 0.0
for r in others:
    for i in snap_idx:
        d = heads[r]["h"][i] - heads[ref]["h"][i]
        dmax = max(dmax, np.nanmax(np.abs(d)))
dmax = float(np.ceil(dmax * 20) / 20)      # round up to a 0.05 ft step
hmin = float(np.nanmin([heads[ref]["h"][i] for i in snap_idx]))
hmax = float(np.nanmax([heads[ref]["h"][i] for i in snap_idx]))
print(f"  head scale {hmin:.2f}..{hmax:.2f} ft;  difference scale +/-{dmax:.2f} ft")

with styles.USGSMap():
    fig, axs = plt.subplots(nrow, 3, figsize=(7.48, 3.1 * nrow), layout="constrained")
    axs = np.atleast_2d(axs)

    for c, (i, lbl) in enumerate(zip(snap_idx, snap_lbl)):
        im = axs[0, c].imshow(heads[ref]["h"][i], cmap="viridis", vmin=hmin, vmax=hmax)
        axs[0, c].set_title(lbl, size=8)
        if c == 0:
            axs[0, c].set_ylabel(f"{res_label[ref]}\n(reference)", size=8)
    cb = fig.colorbar(im, ax=axs[0, :], shrink=0.85, location="right")
    cb.set_label("Layer-1 head, ft", size=8)

    for r_i, r in enumerate(others, start=1):
        for c, i in enumerate(snap_idx):
            d = heads[r]["h"][i] - heads[ref]["h"][i]
            imd = axs[r_i, c].imshow(d, cmap="RdBu_r", vmin=-dmax, vmax=dmax)
            if c == 0:
                axs[r_i, c].set_ylabel(f"{res_label[r]} - {res_label[ref]}", size=8)
        cbd = fig.colorbar(imd, ax=axs[r_i, :], shrink=0.85, location="right")
        cbd.set_label("Head difference, ft", size=8)

    for ax in axs.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"MODFLOW layer-1 heads vs D-Flow FM grid "
                 f"({mf_couple_freq_hours:g} h coupling, n={n_connections})", fontsize=10)
    fig.savefig(fig_ws / f"grid_head_comparison_{mf_couple_freq_hours:g}h{fig_ext}",
                dpi=300, transparent=transparent)
print("wrote", fig_ws / f"grid_head_comparison_{mf_couple_freq_hours:g}h{fig_ext}")

#### Zoomed to the Greenport sewer area

Same panels cropped to the junction window, with the connection cells marked.
The color scale is recomputed over the window so local structure is not
flattened by a whole-domain outlier.

In [ ]:
zmax = 0.0
for r in others:
    for i in snap_idx:
        d = (heads[r]["h"][i] - heads[ref]["h"][i])[r0:r1 + 1, c0:c1 + 1]
        zmax = max(zmax, np.nanmax(np.abs(d)))
zmax = float(np.ceil(zmax * 20) / 20)
zhmin = float(np.nanmin([heads[ref]["h"][i][r0:r1 + 1, c0:c1 + 1] for i in snap_idx]))
zhmax = float(np.nanmax([heads[ref]["h"][i][r0:r1 + 1, c0:c1 + 1] for i in snap_idx]))
print(f"  window head scale {zhmin:.2f}..{zhmax:.2f} ft;  difference +/-{zmax:.2f} ft")

mr, mc = j_row - r0, j_col - c0          # junction markers in window coordinates

with styles.USGSMap():
    fig, axs = plt.subplots(nrow, 3, figsize=(7.48, 3.4 * nrow), layout="constrained")
    axs = np.atleast_2d(axs)

    for c, (i, lbl) in enumerate(zip(snap_idx, snap_lbl)):
        im = axs[0, c].imshow(heads[ref]["h"][i][r0:r1 + 1, c0:c1 + 1],
                              cmap="viridis", vmin=zhmin, vmax=zhmax)
        axs[0, c].set_title(lbl, size=8)
        if c == 0:
            axs[0, c].set_ylabel(res_label[ref] + " (reference)", size=8)
    cb = fig.colorbar(im, ax=axs[0, :], shrink=0.85, location="right")
    cb.set_label("Layer-1 head, ft", size=8)

    for r_i, r in enumerate(others, start=1):
        for c, i in enumerate(snap_idx):
            d = (heads[r]["h"][i] - heads[ref]["h"][i])[r0:r1 + 1, c0:c1 + 1]
            imd = axs[r_i, c].imshow(d, cmap="RdBu_r", vmin=-zmax, vmax=zmax)
            if c == 0:
                axs[r_i, c].set_ylabel(res_label[r] + " - " + res_label[ref], size=8)
        cbd = fig.colorbar(imd, ax=axs[r_i, :], shrink=0.85, location="right")
        cbd.set_label("Head difference, ft", size=8)

    for ax in axs.ravel():
        ax.plot(mc, mr, ls="none", marker="o", ms=2.0, mfc="none",
                mec="black", mew=0.4, alpha=0.8)
        ax.set_xticks([]); ax.set_yticks([])
    axs[0, 0].plot([], [], ls="none", marker="o", ms=4, mfc="none", mec="black",
                   label=f"{n_conn} SWMM connections")
    axs[0, 0].legend(loc="upper left", fontsize=6, framealpha=0.85)
    fig.suptitle("Greenport sewer area - layer-1 heads vs D-Flow FM grid "
                 f"({mf_couple_freq_hours:g} h coupling)", fontsize=10)
    fig.savefig(fig_ws / f"grid_head_comparison_zoom_{mf_couple_freq_hours:g}h{fig_ext}",
                dpi=300, transparent=transparent)
print("wrote", fig_ws / f"grid_head_comparison_zoom_{mf_couple_freq_hours:g}h{fig_ext}")

#### Difference and correlation statistics

Computed over active cells only. `bias` is the mean signed difference, so a
negative value means that grid simulates lower heads than the reference.

In [ ]:
def stats(a, b):
    """Difference statistics of a against reference b, over cells active in both."""
    m = np.isfinite(a) & np.isfinite(b)
    if not m.any():
        return None
    d = a[m] - b[m]
    x, y = b[m], a[m]
    denom = np.sum((x - x.mean()) ** 2)
    return {
        "n": int(m.sum()),
        "bias ft": d.mean(),
        "MAE ft": np.abs(d).mean(),
        "RMSE ft": np.sqrt((d ** 2).mean()),
        "max|d| ft": np.abs(d).max(),
        "pearson r": np.corrcoef(x, y)[0, 1],
        "R2": 1.0 - np.sum((y - x) ** 2) / denom if denom > 0 else np.nan,
    }


# Three scopes. Whole-domain numbers average over inland cells where the grids
# barely differ, so they understate what the sewer actually sees; the junction
# cells are what set the exchange.
def scoped(a, b, scope):
    if scope == "domain":
        return a, b
    if scope == "window":
        return a[..., r0:r1 + 1, c0:c1 + 1], b[..., r0:r1 + 1, c0:c1 + 1]
    return a[..., j_mask], b[..., j_mask]


rows = []
for scope in ("domain", "window", "junctions"):
    for r in others:
        for i, lbl in zip(snap_idx, snap_lbl):
            a, b = scoped(heads[r]["h"][i], heads[ref]["h"][i], scope)
            rows.append({"scope": scope, "grid": res_label[r],
                         "when": lbl.split("(")[0].strip(), **stats(a, b)})
        a, b = scoped(heads[r]["h"][:ntime], heads[ref]["h"][:ntime], scope)
        rows.append({"scope": scope, "grid": res_label[r], "when": "ALL times",
                     **stats(a, b)})

df = pd.DataFrame(rows).set_index(["scope", "grid", "when"])
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
df

#### How the difference evolves

Three snapshots can hide drift or a tidal cycle, so this tracks the statistics
across every time step.

In [ ]:
with styles.USGSMap():
    fig, axs = plt.subplots(2, 1, figsize=(7.5, 6.0), layout="constrained", sharex=True)
    colors = list(mcolors.TABLEAU_COLORS.values())
    for k, r in enumerate(others):
        a, b = heads[r]["h"][:ntime], heads[ref]["h"][:ntime]
        m = np.isfinite(a) & np.isfinite(b)
        bias = np.array([np.nanmean(a[i][m[i]] - b[i][m[i]]) for i in range(ntime)])
        rmse = np.array([np.sqrt(np.nanmean((a[i][m[i]] - b[i][m[i]]) ** 2))
                         for i in range(ntime)])
        t = heads[ref]["t"][:ntime]
        axs[0].plot(t, bias, lw=0.8, color=colors[k], label=f"{res_label[r]} - {res_label[ref]}")
        axs[1].plot(t, rmse, lw=0.8, color=colors[k], label=f"{res_label[r]} - {res_label[ref]}")
    axs[0].axhline(0.0, lw=0.5, ls="--", color="black")
    axs[0].set_ylabel("Bias, ft")
    axs[1].set_ylabel("RMSE, ft")
    axs[1].set_xlabel("Time, days")
    styles.heading(axs[0], heading="Mean signed head difference")
    styles.heading(axs[1], heading="RMS head difference")
    styles.graph_legend(ax=axs[0], loc="best", title="none")
    fig.savefig(fig_ws / f"grid_head_difference_timeseries_{mf_couple_freq_hours:g}h{fig_ext}",
                dpi=300, transparent=transparent)
print("wrote", fig_ws / f"grid_head_difference_timeseries_{mf_couple_freq_hours:g}h{fig_ext}")

#### At the sewer junctions

The whole-domain statistics average over cells far from the coast, where the
grids barely differ. What actually drives the sewer exchange is the head at the
244 connection cells, so those are broken out separately &mdash; along with how
many of them sit above their pipe invert, which is what sets whether a junction
exchanges water at all.

In [ ]:
rc = list(zip(j_row, j_col))      # loaded with the zoom window above
inv = j_inv

rows = []
for r in have:
    hh = heads[r]["h"][:ntime]
    hj = np.array([[hh[i][row, col] for row, col in rc] for i in range(ntime)])
    conn = hj > inv
    rows.append({
        "grid": res_label[r],
        "mean head ft": np.nanmean(hj),
        "connected %": 100.0 * conn.mean(),
        "mean driving head ft": np.nanmean(np.where(conn, hj - inv, np.nan)),
    })
    if r != ref:
        hr = np.array([[heads[ref]["h"][i][row, col] for row, col in rc]
                       for i in range(ntime)])
        rows[-1]["bias vs ref ft"] = np.nanmean(hj - hr)
        rows[-1]["RMSE vs ref ft"] = np.sqrt(np.nanmean((hj - hr) ** 2))

pd.DataFrame(rows).set_index("grid")